# Prompt Testing and Iteration

When a prompt does not behave the way you want, it is tempting to keep tweaking it and rerunning a single example until it looks right. That approach hides problems: a change that fixes one case often quietly breaks another.

This notebook takes a more reliable approach: you build a small set of test cases, run each prompt variant against all of them, and compare the results, so you can see what a change actually does before you commit to it. You will follow this loop throughout:

```mermaid
flowchart LR
    A[Task] --> B[Test cases]
    B --> C[Prompt variant]
    C --> D[Run chain]
    D --> E[Inspect outputs]
    E --> F[Revise prompt]
    F --> C
```

## Learning Objectives

By the end of this notebook, you should be able to:

- Build a small evaluation set that exposes a prompt's weak spots.
- Compare a baseline prompt against a stricter one on the same cases.
- Apply few-shot examples to improve consistency on harder cases.
- Reuse this prompt-testing loop in your own projects.

In [ ]:
# Keep the setup cell focused so the notebook is easy to rerun from the top.
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate


def print_outputs(cases: list[dict[str, str]], outputs: list[str]) -> None:
    for case, output in zip(cases, outputs):
        print("=" * 100)
        print(f"Case: {case['label']}")
        print(f"What to watch: {case['watch_for']}")
        print(output)

In [ ]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=512,
)

output_parser = StrOutputParser()

## Step 1 - Create a Tiny Evaluation Set

For prompt testing, a tiny but varied test set is often more useful than one perfect example. The goal is to include a few cases that are different enough to expose prompt weaknesses.

In this notebook, we will use a simple support-triage task. Each case is a short customer message, and we want the model to:
- estimate urgency
- identify the topic
- suggest a short response


In [ ]:
# Keep the test set small enough to inspect manually.
support_cases = [
    {
        "label": "Damaged package",
        "message": "My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.",
        "watch_for": "Should likely be high urgency and mention replacement or support steps.",
    },
    {
        "label": "Password reset",
        "message": "I cannot sign in to my account because I forgot my password. How do I reset it?",
        "watch_for": "Should be account-related and not overstate urgency.",
    },
    {
        "label": "Double charge",
        "message": "I think my card was charged twice for the same order. Can someone check this?",
        "watch_for": "Should identify billing and treat it as fairly urgent.",
    },
    {
        "label": "Sustainability question",
        "message": "Your website says your products are sustainable. What materials do you actually use?",
        "watch_for": "Should stay low urgency and answer as a product question, not a complaint.",
    },
]

support_cases

A compact evaluation set like this helps you ask practical questions:
- Which cases produce messy or inconsistent output?
- Which cases are misclassified?
- Which instructions are too vague?

Notice that the test set mixes logistics, account access, billing, and product questions. That variety makes the prompt easier to stress-test.

## Step 2 - Start With a Baseline Prompt

We begin with a deliberately simple prompt. It is good enough to run, but it leaves a lot of room for the model to choose its own output structure.

In [ ]:
# This baseline prompt defines the task but does not strongly constrain the format.
baseline_template = """You are helping an ecommerce support team.
Read the customer message and identify the urgency, the topic, and a short reply.

Customer message:
{message}
"""

baseline_prompt = PromptTemplate(
    template=baseline_template,
    input_variables=["message"],
)

baseline_chain = baseline_prompt | llm | output_parser

In [ ]:
# Try the baseline prompt on one case before scaling up.
print(baseline_chain.invoke({"message": support_cases[0]["message"]}))

## Step 3 - Evaluate the Baseline Across All Cases

One output is not enough to judge a prompt. Prompt testing becomes more informative when you run the same prompt on several cases and compare the outputs side by side.

In [ ]:
# batch() lets us reuse the same prompt on many cases at once.
baseline_inputs = [{"message": case["message"]} for case in support_cases]
baseline_outputs = baseline_chain.batch(baseline_inputs)
print_outputs(support_cases, baseline_outputs)

### What Usually Goes Wrong in the Baseline

With a loose prompt, common issues include:
- inconsistent formatting from one case to another
- uneven detail across cases
- vague urgency labels
- responses that are friendly but hard to compare systematically

This is a good sign that we need a stronger prompt specification, not necessarily a different model.

## Step 4 - Improve the Prompt With Stronger Instructions

Now we make the task more explicit. We still use the same model, but we add tighter rules for the output format so the results become easier to inspect and compare.

In [ ]:
# Stronger instructions often improve consistency more than changing the model does.
improved_template = """You are triaging ecommerce support messages.
Analyze the customer message and return exactly this format:

Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Keep the reply concise and practical.

Customer message:
{message}
"""

improved_prompt = PromptTemplate(
    template=improved_template,
    input_variables=["message"],
)

improved_chain = improved_prompt | llm | output_parser

In [ ]:
# Compare the improved prompt on the same evaluation set.
improved_outputs = improved_chain.batch(baseline_inputs)
print_outputs(support_cases, improved_outputs)

#### Compare Two Prompt Variants on One Hard Case

Comparing full test sets is useful, but sometimes you also want to zoom in on one case that feels tricky and inspect how each prompt handles it.

In [ ]:
hard_case = support_cases[2]["message"]

print("BASELINE")
print(baseline_chain.invoke({"message": hard_case}))
print("\n" + "-" * 100 + "\n")
print("IMPROVED")
print(improved_chain.invoke({"message": hard_case}))

## Step 5 - Add Few-Shot Examples for Harder Edge Cases

If the output is still inconsistent, few-shot examples can teach the model the exact style you want. This is especially useful when the format matters or when the task includes edge cases that the model may interpret differently.

```mermaid
flowchart LR
    A[Example input-output pairs] --> B[FewShotPromptTemplate]
    C[New customer message] --> B
    B --> D[Formatted prompt]
    D --> E[More controlled output]
```

The examples do not retrain the model. They just give the model a clearer pattern to imitate for the current request.

In [ ]:
# Each example demonstrates the output style we want on a realistic support case.
examples = [
    {
        "message": "My order still has not arrived and I need it tomorrow for an event.",
        "analysis": "Urgency: high\nTopic: shipping\nSuggested reply:\n- I am sorry your order has not arrived in time for your event.\n- Please send us your order number so we can check shipping options right away.",
    },
    {
        "message": "I forgot my password and cannot get into my account.",
        "analysis": "Urgency: medium\nTopic: account\nSuggested reply:\n- Please use the password reset link on the sign-in page to create a new password.\n- If that does not work, contact support and we can help verify your account.",
    },
]

example_prompt = PromptTemplate(
    input_variables=["message", "analysis"],
    template="""Example message:\n{message}\n\nExample output:\n{analysis}""",
)

prefix = """You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance."""

suffix = """Now analyze this customer message:
{message}"""

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain = few_shot_prompt | llm | output_parser

#### Inspect the Final Few-Shot Prompt

When working with more complex prompt templates, printing the final prompt is a great debugging habit. It lets you check whether the examples and the new input were assembled the way you expected.

In [ ]:
print(few_shot_prompt.format(message=support_cases[0]["message"]))

In [ ]:
# Evaluate the few-shot prompt on the same test set.
few_shot_outputs = few_shot_chain.batch(baseline_inputs)
print_outputs(support_cases, few_shot_outputs)

## Step 6 - Reuse the Prompt on a New Test Set

A prompt is more convincing when it generalizes beyond the first set of examples. Here we apply the same few-shot prompt to a second mini test set with different customer messages.

In [ ]:
new_cases = [
    {
        "label": "Gift card request",
        "message": "Do you sell gift cards? I want to send one to a friend for their birthday.",
        "watch_for": "Should be product-related and low urgency.",
    },
    {
        "label": "Express shipping",
        "message": "I placed my order an hour ago. Can I upgrade it to express shipping?",
        "watch_for": "Should likely be shipping-related and time-sensitive.",
    },
]

new_outputs = few_shot_chain.batch([{"message": case["message"]} for case in new_cases])
print_outputs(new_cases, new_outputs)

## Exercises

1. Add one more support case to `support_cases` and test all three prompt variants.
2. Change the improved prompt so it must return JSON-like output instead of bullet points.
3. Add one more few-shot example for a returns-related message and compare the result.
4. Raise the model temperature and check whether output consistency gets better or worse.


### Exercise 1: Add a support case and test all three prompt variants

In [ ]:
# Add a new case, rebuild the inputs, then run all three chains on the full set.
support_cases.append(
    {
        "label": "Wrong item",
        "message": "I ordered a blue jacket but received a red one. How do I exchange it?",
        "watch_for": "Should be returns related and around medium urgency.",
    }
)

cases_inputs = [{"message": case["message"]} for case in support_cases]
for name, chain in [
    ("baseline", baseline_chain),
    ("improved", improved_chain),
    ("few-shot", few_shot_chain),
]:
    print("#" * 30, name, "#" * 30)
    print_outputs(support_cases, chain.batch(cases_inputs))

**Explanation**

Appending to `support_cases` and rebuilding the input list lets you run the new message through every prompt variant on the same footing. Running all three chains side by side is the point of a test set: you can see whether the new case is classified consistently, and where the baseline, improved, and few-shot prompts disagree.

### Exercise 2: Make the improved prompt return JSON-like output

In [ ]:
# Ask for a fixed JSON object instead of bullet points.
# In a PromptTemplate, literal braces must be doubled so they are not read as variables.
json_template = """You are triaging ecommerce support messages.
Analyze the customer message and return only a JSON object with exactly these keys:
{{"urgency": "low|medium|high", "topic": "shipping|returns|billing|account|product", "reply": "a short practical reply"}}

Customer message:
{message}
"""

json_prompt = PromptTemplate(template=json_template, input_variables=["message"])
json_chain = json_prompt | llm | output_parser

print(json_chain.invoke({"message": support_cases[0]["message"]}))

**Explanation**

The braces around the JSON object are doubled (`{{ }}`) so `PromptTemplate` treats them as literal text rather than input variables, leaving `{message}` as the only variable. A fixed JSON shape is easier to validate and parse downstream than free-form bullets, though you should still guard against the model adding stray text around the object.

### Exercise 3: Add a returns few-shot example and compare

In [ ]:
# Add a returns example, then compare the two-example and three-example prompts on a returns message.
returns_example = {
    "message": "I returned my shoes last week but I have not received my refund yet.",
    "analysis": "Urgency: medium\nTopic: returns\nSuggested reply:\n- Thanks for your patience, refunds usually take a few business days after we receive the return.\n- Please share your order number so we can confirm the refund status.",
}

few_shot_prompt_v2 = FewShotPromptTemplate(
    examples=examples + [returns_example],
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)
few_shot_chain_v2 = few_shot_prompt_v2 | llm | output_parser

test_message = "My package was damaged and I want to send it back for a refund."
print("two examples:\n", few_shot_chain.invoke({"message": test_message}))
print(
    "\nthree examples (with returns):\n",
    few_shot_chain_v2.invoke({"message": test_message}),
)

**Explanation**

Adding a returns example gives the model a concrete template for refund-style messages. Comparing the original two-example chain with the three-example chain on the same returns case shows whether the extra demonstration sharpens the topic label and the reply. More examples usually help on the covered case but also lengthen the prompt, so add them where they earn their place.

### Exercise 4: Raise the temperature and check consistency

In [ ]:
# Run the same case several times at a higher temperature to see how much the output varies.
hot_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=1.0, max_tokens=512)
hot_chain = improved_prompt | hot_llm | output_parser

message = support_cases[1]["message"]
for run in range(3):
    print("#" * 20, f"run {run + 1}", "#" * 20)
    print(hot_chain.invoke({"message": message}))

**Explanation**

Temperature controls sampling randomness. The lesson uses 0.1, which is near-deterministic and good for repeatable testing. At temperature 1.0 the same input usually varies more between runs, which is the opposite of what you want when comparing prompts: the differences should come from the prompt, not from sampling noise. Keep the temperature low for prompt testing, and raise it only when you actively want more varied output.

## Conclusion

In this notebook, you practiced a simple but powerful prompt-engineering workflow:
- define a task clearly
- create a compact evaluation set
- compare prompt variants on the same cases
- tighten the prompt format when outputs are inconsistent
- add few-shot examples when you need even more control

This kind of lightweight testing loop is a strong next step after learning the LangChain basics and the core prompt-engineering patterns.